# SolarSentinel AI - 02: VGG16 Transfer Learning & Optimization

This notebook demonstrates the end-to-end training pipeline for the VGG16 backbone on solar panel defect classification.

### Topics Covered:
1. Loading the pre-trained ImageNet VGG16 backbone.
2. Architecture design: Head A (GlobalAveragePooling2D) vs Head B (Dense + BatchNormalization).
3. ImageNet mean subtraction and domain-specific data augmentation.
4. Two-stage training: Feature Extraction (Backbone frozen) followed by Block 5 Fine-Tuning.

In [ ]:
import os
import sys
import tensorflow as tf
from tensorflow.keras.applications import VGG16

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from ml.models.vgg16 import build_vgg16_classifier, print_model_summary
from ml.data.preprocessing import load_dataset_from_manifest, build_augmentation_layer

## 1. Comparing Head Architectures

In [ ]:
model_a = build_vgg16_classifier(head_type="head_a", freeze_backbone=True)
print("=== Head A (Compact GAP Head) ===")
print_model_summary(model_a)

model_b = build_vgg16_classifier(head_type="head_b", freeze_backbone=True)
print("\n=== Head B (Deep Dense + BatchNormalization Head) ===")
print_model_summary(model_b)

## 2. Preprocessing & Augmentation Pipeline

In [ ]:
aug_pipeline = build_augmentation_layer()
print("Data Augmentation Operations:")
for layer in aug_pipeline.layers:
    print(f" - {layer.name}: {layer.get_config()}")

## 3. Two-Stage Fine-Tuning Strategy

In [ ]:
print("Stage 1: Frozen Backbone (14,714,688 non-trainable weights, training only classification head)")
print("Stage 2: Differential Fine-Tuning (Unfreezing Block 5 conv layers with lr=1e-5)")